# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [ ]:
import pandas as pd
 
# 1. Load all three raw files
URL1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
URL2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
URL3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"
 
df1 = pd.read_csv(URL1)
df2 = pd.read_csv(URL2)
df3 = pd.read_csv(URL3)
 
print(f"Raw shapes → file1: {df1.shape}, file2: {df2.shape}, file3: {df3.shape}")
 
 
# 2. Standardise column names before combining
COLUMN_MAP = {
    "ST": "state",
    "State": "state",
    "GENDER": "gender",
    "Gender": "gender",
    "Customer": "customer",
    "Education": "education",
    "Customer Lifetime Value": "customer_lifetime_value",
    "Income": "income",
    "Monthly Premium Auto": "monthly_premium_auto",
    "Number of Open Complaints": "number_of_open_complaints",
    "Policy Type": "policy_type",
    "Vehicle Class": "vehicle_class",
    "Total Claim Amount": "total_claim_amount",
}
 
df1.rename(columns=COLUMN_MAP, inplace=True)
df2.rename(columns=COLUMN_MAP, inplace=True)
df3.rename(columns=COLUMN_MAP, inplace=True)
 
 
# 3. Combine all three datasets
df = pd.concat([df1, df2, df3], ignore_index=True)
print(f"Combined shape (before cleaning): {df.shape}")
 
 
# 4. Cleaning function
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
 
    # 4a. Drop rows where every cell is NaN (empty rows from file1 tail)
    df.dropna(how="all", inplace=True)
 
    # 4b. Drop the 'customer' column – it is just an ID, not useful for analysis
    #        (keep if you need it; comment this line out)
    # df.drop(columns=["customer"], inplace=True)
 
    # 4c. Standardise 'state' values
    state_map = {
        "Cali": "California",
        "AZ": "Arizona",
        "WA": "Washington",
    }
    df["state"] = df["state"].replace(state_map)
 
    # 4d. Standardise 'gender' values → 'M' / 'F'
    gender_map = {
        "Male": "M",
        "male": "M",
        "Femal": "F",
        "female": "F",
        "Female": "F",
    }
    df["gender"] = df["gender"].replace(gender_map)
 
    # 4e. Clean 'customer_lifetime_value' – some values end with '%'
    df["customer_lifetime_value"] = (
        df["customer_lifetime_value"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .replace("nan", pd.NA)
    )
    df["customer_lifetime_value"] = pd.to_numeric(
        df["customer_lifetime_value"], errors="coerce"
    )
 
    # 4f. Clean 'number_of_open_complaints' – stored as '1/5/00' fractions in file1
    #         Extract the numerator (first digit before '/')
    df["number_of_open_complaints"] = (
        df["number_of_open_complaints"]
        .astype(str)
        .str.split("/")
        .str[0]
        .replace("nan", pd.NA)
    )
    df["number_of_open_complaints"] = pd.to_numeric(
        df["number_of_open_complaints"], errors="coerce"
    )
 
    # 4g. Ensure numeric types for remaining numeric columns
    numeric_cols = ["income", "monthly_premium_auto", "total_claim_amount"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
 
    # 4h. Strip leading/trailing whitespace from all string columns
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
 
    # 4i. Reset index after dropping rows
    df.reset_index(drop=True, inplace=True)
 
    return df
 
 
# 5. Apply cleaning
df_clean = clean_dataframe(df)
 
print(f"Cleaned shape: {df_clean.shape}")
print("\nColumn dtypes after cleaning:")
print(df_clean.dtypes)
print("\nNull counts per column:")
print(df_clean.isnull().sum())
print("\nSample rows:")
print(df_clean.head())
 
 
# 6. Save the combined clean dataset
OUTPUT_PATH = "insurance_combined_clean.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"\nCleaned dataset saved to '{OUTPUT_PATH}'")

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")

pivot1 = df.pivot_table(
    values="total_claim_amount",
    index="sales_channel",
    aggfunc="sum"
).round(2).sort_values("total_claim_amount", ascending=False)

pivot1.columns = ["total_revenue"]
print(pivot1)

pivot2 = df.pivot_table(
    values="customer_lifetime_value",
    index="education",
    columns="gender",
    aggfunc="mean"
).round(2)

print(pivot2)

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")

# Step 1: pivot table (wide format) — sum of complaints by policy type × month
pivot = df.pivot_table(
    values="number_of_open_complaints",
    index="policy_type",
    columns="month",
    aggfunc="sum"
).round(2)

# Step 2: melt to long format — one row per (policy_type, month) combination
long_df = pivot.reset_index().melt(
    id_vars="policy_type",
    var_name="month",
    value_name="number_of_complaints"
)

long_df = long_df.sort_values(["month", "policy_type"]).reset_index(drop=True)
print(long_df)